In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
from fussion_gauss import *
from project import *
from obtain_allocation import *
from generate_gaussian import *
import time
from datetime import datetime

# Estudio de las distribuciones de los tiempos entre paradas.

For realizing this study, we have to study, for each line the data in a specific timeslot between the n-th and the n+1-th stop. Then if succeed, export and expand this analysis to a batch study for every stop intersection

In [ ]:
list_stops_toA = [2646, 562, 4090, 4188, 5939, 3995, 3991, 3903, 2539, 2535, 1929, 4032, 5360, 82, 5727]

E01_toA = {}


for stop in list_stops_toA:
    data_stop = pd.read_csv(f"../data/E01_toA/NEW/clean_data_newnew_{stop}.csv", encoding="utf-8", header=0)
    data_stop = clean_dataframe(data_stop, 2)
    data_stop = clean_dataframe_advanced(data_stop, "LA")
    E01_toA[stop] = data_stop


Once saved. Study 2 stops. Extract all the occurences between FE 562 and 4090 for TimeSlot 10, then match the tripID  to locate same bus, and save the timebetween stops. Make a distribution for that and study its topology

In [ ]:

data_first_stop = E01_toA[list_stops_toA[1]]

data_second_stop = E01_toA[list_stops_toA[2]]

data_first_stop = data_first_stop[data_first_stop["TimeSlot.1"] == 10]
data_second_stop = data_second_stop[data_second_stop["TimeSlot.1"] == 10]

#viva la fuerza bruta

list_tuples = []
for idx, row in data_first_stop.iterrows():
    for idx_2, row_2 in data_second_stop.iterrows():
        if row.TripId == row_2.TripId:
            list_tuples.append((row, row_2))

len(list_tuples)


#Needede to remove the last element(order of nanoseconds) from the EventTime
def parse_time(s):
    return datetime.strptime(s[:15], "%H:%M:%S.%f")


distribution_data = []
for element in list_tuples:
    distribution_data.append((parse_time(element[1]["EventTime.1"]) - parse_time(element[0]["EventTime.1"])).total_seconds())


distribution_data

In [ ]:
distribution_data.sort()

is_normal, inst, mean, std = is_normal_simplified(distribution_data)

plt.hist(distribution_data, bins= 15)
plt.xlabel("Tiempo entre paradas")
plt.ylabel('Count')
plt.title(f"Histograma de Tiempo entre paradas")
plt.xticks(rotation=29)
plt.grid(True, alpha=0.3)
plt.show()

Una normal preciosa si se elimina el ultimmo valor(Un outlier como la copa de un pino)

Expandimos a todas las paradas en esta franja primero, para ver si 1. Son normales(deberian) y 2. si no lo son, viendo si mandando a tomar por culo sus outliers lo son

In [ ]:

counter = len(list_stops_toA)


global_normals = 0
counter_global = 0

for iteration in range(0, counter -1):
    data_first_stop = E01_toA[list_stops_toA[iteration]]
    data_second_stop = E01_toA[list_stops_toA[iteration + 1]]
    data_first_stop = data_first_stop[data_first_stop["TimeSlot.1"] == 10]
    data_second_stop = data_second_stop[data_second_stop["TimeSlot.1"] == 10]

    #viva la fuerza bruta

    list_tuples = []
    for idx, row in data_first_stop.iterrows():
        for idx_2, row_2 in data_second_stop.iterrows():
            if row.TripId == row_2.TripId:
                list_tuples.append((row, row_2))

    distribution_data = []
    for element in list_tuples:
        distribution_data.append((parse_time(element[1]["EventTime.1"]) - parse_time(element[0]["EventTime.1"])).total_seconds())

    distribution_data.sort()

    is_normal, inst, mean, std = is_normal_simplified(distribution_data)

    print(f"For the iteration {iteration} the assumtion is {is_normal}")
    plt.hist(distribution_data, bins= 15)
    plt.xlabel("Tiempo entre paradas")
    plt.ylabel('Count')
    plt.title(f"Histograma de Tiempo entre paradas")
    plt.xticks(rotation=29)
    plt.grid(True, alpha=0.3)
    plt.show()

    if is_normal:
        global_normals += 1

    if not is_normal:
        out_list = estimate_outliers(distribution_data.copy(), 20, False)

        for item in out_list:
            distribution_data.remove(item)





        is_normal, inst, mean, std = is_normal_simplified(distribution_data)

        if is_normal:
            global_normals += 1
        print(f"Now after not being normal is now {is_normal} without {len(out_list)} outliers")
        plt.hist(distribution_data, bins= 15, color='red')
        plt.xlabel("Tiempo entre paradas")
        plt.ylabel('Count')
        plt.title(f"Histograma de Tiempo entre paradas")
        plt.xticks(rotation=29)
        plt.grid(True, alpha=0.3)
        plt.show()

    counter_global += 1



In [ ]:
print(f"accuracy of {(global_normals/counter_global)*100} %")

Having 100% accuracy. Nice. Now do this for EVERY timeslot

In [ ]:

counter = len(list_stops_toA)

global_normals = 0
counter_global = 0

for iteration in range(0, counter -1):

    for timeslot in range(5, 24):
        data_first_stop = E01_toA[list_stops_toA[iteration]]
        data_second_stop = E01_toA[list_stops_toA[iteration + 1]]
        data_first_stop = data_first_stop[data_first_stop["TimeSlot.1"] == timeslot]
        data_second_stop = data_second_stop[data_second_stop["TimeSlot.1"] == timeslot]

        #viva la fuerza bruta

        list_tuples = []
        for idx, row in data_first_stop.iterrows():
            for idx_2, row_2 in data_second_stop.iterrows():
                if row.TripId == row_2.TripId:
                    list_tuples.append((row, row_2))

        distribution_data = []
        for element in list_tuples:
            distribution_data.append((parse_time(element[1]["EventTime.1"]) - parse_time(element[0]["EventTime.1"])).total_seconds())

        distribution_data.sort()

        is_normal, inst, mean, std = is_normal_simplified(distribution_data)

        print(f"For the iteration {iteration} the assumtion is {is_normal}")
        plt.hist(distribution_data, bins= 15)
        plt.xlabel("Tiempo entre paradas")
        plt.ylabel('Count')
        plt.title(f"Histograma de Tiempo entre paradas")
        plt.xticks(rotation=29)
        plt.grid(True, alpha=0.3)
        plt.show()

        if is_normal:
            global_normals += 1

        if not is_normal:
            out_list = estimate_outliers(distribution_data.copy(), 20, False)

            for item in out_list:
                distribution_data.remove(item)





            is_normal, inst, mean, std = is_normal_simplified(distribution_data)

            if is_normal:
                global_normals += 1
            print(f"Now after not being normal is now {is_normal} without {len(out_list)} outliers")
            plt.hist(distribution_data, bins= 15, color='red')
            plt.xlabel("Tiempo entre paradas")
            plt.ylabel('Count')
            plt.title(f"Histograma de Tiempo entre paradas")
            plt.xticks(rotation=29)
            plt.grid(True, alpha=0.3)
            plt.show()

        counter_global += 1



In [ ]:
print(f"accuracy of {(global_normals/counter_global)*100} %")

With a general accuracy of 95.48% we can assume that the distributions for every Timeslot for every intersection between stops can be represented as a normal distribution.

In [43]:

counter = len(list_stops_toA)

global_normals = 0
counter_global = 0




columns_times_stops = []
columns_times_stops.append("StopsCodes")

#generate columns. The columns are gonna be the timeslots with Mean and Std as all are normals

for timeslot in range(5, 24):
    columns_times_stops.append(f"Mean_{timeslot}")
    columns_times_stops.append(f"Std_{timeslot}")

new_dataframe = pd.DataFrame(columns=columns_times_stops)




for iteration in range(0, counter -1):
    #cleanup of the list buffer
    new_row = []

    new_row.append(f"{list_stops_toA[iteration]}-{list_stops_toA[iteration + 1]}")
    for timeslot in range(5, 24):



        data_first_stop = E01_toA[list_stops_toA[iteration]]
        data_second_stop = E01_toA[list_stops_toA[iteration + 1]]
        data_first_stop = data_first_stop[data_first_stop["TimeSlot.1"] == timeslot]
        data_second_stop = data_second_stop[data_second_stop["TimeSlot.1"] == timeslot]

        #viva la fuerza bruta
        list_tuples = []
        for idx, row in data_first_stop.iterrows():
            for idx_2, row_2 in data_second_stop.iterrows():
                if row.TripId == row_2.TripId:
                    list_tuples.append((row, row_2))

        distribution_data = []
        for element in list_tuples:
            distribution_data.append((parse_time(element[1]["EventTime.1"]) - parse_time(element[0]["EventTime.1"])).total_seconds())

        distribution_data.sort()

        is_normal, inst, mean, std = is_normal_simplified(distribution_data)

        if is_normal:
            new_row.append(mean)
            new_row.append(std)

            continue


        if not is_normal:
            out_list = estimate_outliers(distribution_data.copy(), 20, False)

            for item in out_list:
                distribution_data.remove(item)

            is_normal, inst, mean, std = is_normal_simplified(distribution_data)
            new_row.append(mean)
            new_row.append(std)

    new_dataframe.loc[len(new_dataframe)] = new_row

new_dataframe

,StopsCodes,Mean_5,Std_5,Mean_6,Std_6,Mean_7,Std_7,Mean_8,Std_8,Mean_9,...,Mean_19,Std_19,Mean_20,Std_20,Mean_21,Std_21,Mean_22,Std_22,Mean_23,Std_23
0,2646-562,293.888111,171.773699,596.962024,233.424005,505.571332,268.350798,480.636569,215.443646,612.676497,...,422.240988,237.862168,475.024627,240.780102,541.177176,271.304594,582.298526,289.842523,250.692971,114.929839
1,562-4090,114.482977,26.038392,127.294168,30.270864,137.384598,35.262211,144.490165,37.947491,149.978611,...,153.215691,38.885485,149.974989,36.362950,134.811308,37.299372,115.873088,30.805581,105.009185,31.203253
2,4090-4188,149.690924,30.344019,152.953386,33.141555,168.233350,34.209274,169.804176,34.398135,176.691819,...,185.567611,39.747351,186.079203,41.202260,171.834726,35.154159,173.513644,37.063752,162.492346,30.628158
3,4188-5939,158.129190,27.837213,162.664755,27.193222,172.967191,33.294624,172.943806,30.280782,179.073719,...,283.262131,71.340123,301.763393,64.887030,314.610406,75.109850,331.231142,94.653575,257.524200,51.170034
4,5939-3995,126.476728,29.651394,138.116262,29.063303,157.652943,37.644228,155.478753,39.783762,152.165368,...,155.807493,38.113649,155.045099,36.617191,143.359238,33.197651,134.842274,29.994995,125.420087,26.881384
5,3995-3991,138.122236,23.895121,152.065746,27.926258,162.365770,35.859565,159.286159,34.674444,150.132842,...,143.195170,31.643617,141.016372,30.747233,137.655543,28.141532,132.264148,28.365726,114.421378,22.373800
6,3991-3903,110.881991,18.168298,112.462246,20.613036,114.827372,24.559886,118.583218,24.706013,113.903343,...,116.214427,22.728858,116.178737,23.355908,110.880094,17.802552,103.999199,16.033555,95.132476,13.757222
7,3903-2539,182.351864,36.262293,196.483838,41.815846,255.542708,80.782955,262.576389,88.630569,246.507505,...,253.184773,77.367419,228.910518,60.297400,205.924738,47.393989,186.581750,36.593275,170.212325,34.761973
8,2539-2535,133.909741,18.618003,153.191481,47.206231,280.315673,136.614202,365.639862,195.565554,295.201824,...,150.889990,34.638825,137.965450,20.682968,125.789265,14.077065,118.142773,13.965065,109.587008,12.547811
9,2535-1929,117.887547,36.862584,142.425356,35.980947,160.775128,41.132789,178.639843,41.849501,172.559170,...,174.932154,39.193113,163.318455,36.876374,148.110585,32.821779,133.933970,30.997108,124.993298,32.652525


In [44]:
with open ("../data/E01_toA/NEW/RESULTS/input_times_stops.csv", "w", encoding="utf-8") as f:
    new_dataframe.to_csv(f, index=False)

## Now repeat the process for the toB route

In [49]:


list_stops_toB = [2646, 562, 4090, 4188, 5939, 3995, 3991, 3903, 2539, 2535, 1929, 4032, 5360, 82, 5727]

list_stops_toB = [5727, 81, 1167, 2538, 3929, 3990, 3994, 5938, 4187, 2964, 649, 2646]

E01_toB = {}


for stop in list_stops_toB:
    data_stop = pd.read_csv(f"../data/E01_toB/clean_data_{stop}.csv", encoding="utf-8", header=0)
    data_stop = clean_dataframe(data_stop, 1)
    data_stop = clean_dataframe_advanced(data_stop, "LA")
    E01_toB[stop] = data_stop






counter = len(list_stops_toB)

global_normals = 0
counter_global = 0




columns_times_stops = []
columns_times_stops.append("StopsCodes")

#generate columns. The columns are gonna be the timeslots with Mean and Std as all are normals

for timeslot in range(5, 24):
    columns_times_stops.append(f"Mean_{timeslot}")
    columns_times_stops.append(f"Std_{timeslot}")

new_dataframe = pd.DataFrame(columns=columns_times_stops)




for iteration in range(0, counter -1):
    #cleanup of the list buffer
    new_row = []

    new_row.append(f"{list_stops_toB[iteration]}-{list_stops_toB[iteration + 1]}")
    for timeslot in range(5, 24):



        data_first_stop = E01_toB[list_stops_toB[iteration]]
        data_second_stop = E01_toB[list_stops_toB[iteration + 1]]
        data_first_stop = data_first_stop[data_first_stop["TimeSlot.1"] == timeslot]
        data_second_stop = data_second_stop[data_second_stop["TimeSlot.1"] == timeslot]

        #viva la fuerza bruta
        list_tuples = []
        for idx, row in data_first_stop.iterrows():
            for idx_2, row_2 in data_second_stop.iterrows():
                if row.TripId == row_2.TripId:
                    list_tuples.append((row, row_2))

        distribution_data = []
        for element in list_tuples:
            distribution_data.append((parse_time(element[1]["EventTime.1"]) - parse_time(element[0]["EventTime.1"])).total_seconds())

        distribution_data.sort()

        is_normal, inst, mean, std = is_normal_simplified(distribution_data)

        if is_normal:
            new_row.append(mean)
            new_row.append(std)

            continue


        if not is_normal:
            out_list = estimate_outliers(distribution_data.copy(), 20, False)

            for item in out_list:
                distribution_data.remove(item)

            is_normal, inst, mean, std = is_normal_simplified(distribution_data)
            new_row.append(mean)
            new_row.append(std)

    new_dataframe.loc[len(new_dataframe)] = new_row


In [50]:
with open ("../data/E01_toB/RESULTS/input_times_stops_toB.csv", "w", encoding="utf-8") as f:
    new_dataframe.to_csv(f, index=False)